## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI:

```bash
az login
```

**Redis Server Required:**
```bash
# Using Docker:
docker run -d -p 6379:6379 redis:latest
```

# 🏦 Redis Chat Message Store for Threads

## Industry Use Case: Distributed Customer Session Management

This notebook demonstrates how to use **Redis** for managing conversation threads across distributed banking applications.

| Feature | FSI Application |
|---------|----------------|
| **Distributed Sessions** | Scale across multiple app instances |
| **Session Persistence** | Customer conversations survive restarts |
| **Message Limits** | Automatic conversation trimming |
| **User Isolation** | Separate threads per customer/session |

In [2]:
import os
from uuid import uuid4
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from agent_framework import Agent, AgentSession
from agent_framework.foundry import FoundryChatClient
from agent_framework.redis import RedisHistoryProvider

load_dotenv('../../.env', override=True)

print(f"✅ Environment loaded")
print(f"Project Endpoint: {os.getenv('AI_FOUNDRY_PROJECT_ENDPOINT')}")

✅ Environment loaded
Project Endpoint: https://demopocaifoundry.services.ai.azure.com/api/projects/demoproject


In [4]:
async def get_account_balance(account_id: str) -> str:
    """Get account balance for a customer."""
    accounts = {
        "ACC-1001": {"balance": 25430.50, "type": "Checking", "currency": "USD"},
        "ACC-1002": {"balance": 150000.00, "type": "Savings", "currency": "USD"},
    }
    a = accounts.get(account_id, {"balance": 0, "type": "Unknown", "currency": "USD"})
    return f"Account {account_id}: {a['type']} - Balance ${a['balance']:,.2f} {a['currency']}"


async def transfer_funds(from_account: str, to_account: str, amount: float) -> str:
    """Transfer funds between accounts (simulated)."""
    return f"Transfer initiated: ${amount:,.2f} from {from_account} to {to_account}. Reference: TXN-{uuid4().hex[:8].upper()}"


print("✅ Tools defined: get_account_balance, transfer_funds")

✅ Tools defined: get_account_balance, transfer_funds


## Example 1: Basic Redis Store

Create a Redis-backed conversation with automatic persistence.

In [5]:
async def example_basic_redis_store():
    """Basic Redis history provider example."""
    print("\n--- Basic Redis History Provider ---\n")
    
    # Create Redis history provider
    redis_provider = RedisHistoryProvider(
        redis_url="redis://localhost:6379",
        ssl=False,
    )
    print(f"✅ Created Redis history provider")
    
    # Create agent using FoundryChatClient with Redis history
    PROJECT_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
    
    client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=AzureCliCredential(),
    )
    agent = Agent(
        client=client,
        name="BankingSupport",
        instructions="You are a helpful banking support agent. Help customers with account inquiries.",
        tools=[get_account_balance, transfer_funds],
        context_providers=[redis_provider],
    )
    
    session = agent.create_session()
    print(f"✅ Session created: {session.session_id}\n")
    
    # Have a conversation
    query1 = "Hello! Can you check my balance for account ACC-1001?"
    print(f"Customer: {query1}")
    response1 = await agent.run(query1, session=session)
    print(f"Agent: {response1.text}\n")
    
    query2 = "Thanks! What about my savings account ACC-1002?"
    print(f"Customer: {query2}")
    response2 = await agent.run(query2, session=session)
    print(f"Agent: {response2.text}\n")
    
    # Check Redis storage
    messages = await redis_provider.get_messages(session.session_id)
    print(f"📊 Messages in Redis: {len(messages)}")
    
    # Cleanup
    await redis_provider.clear(session.session_id)
    await redis_provider.aclose()
    print("✅ Redis data cleaned up\n")

await example_basic_redis_store()


--- Basic Redis History Provider ---

✅ Created Redis history provider
✅ Session created: 19dc5e32-343e-465a-b3e5-1c2f4b9c3ec4

Customer: Hello! Can you check my balance for account ACC-1001?


ConnectionError: Error Multiple exceptions: [Errno 10061] Connect call failed ('::1', 6379, 0, 0), [Errno 10061] Connect call failed ('127.0.0.1', 6379) connecting to localhost:6379.

In [ ]:
## Example 2: User Session Management

Track conversations per customer with isolated session storage.

async def example_user_session_management():
    """User session management with Redis."""
    print("\n--- User Session Management ---\n")
    
    customer_id = "CUST-12345"
    
    # Create Redis history provider with message limit for this customer
    redis_provider = RedisHistoryProvider(
        redis_url="redis://localhost:6379",
        ssl=False,
        key_prefix=f"customer_{customer_id}",
        max_messages=10,  # Keep only last 10 messages
    )
    
    # Create agent with Redis history provider
    PROJECT_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
    
    client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=AzureCliCredential(),
    )
    agent = Agent(
        client=client,
        name="BankingSupport",
        instructions="You are a banking support agent. Remember customer preferences.",
        tools=[get_account_balance, transfer_funds],
        context_providers=[redis_provider],
    )
    
    session = agent.create_session()
    print(f"✅ Session started for customer {customer_id}")
    print(f"   Session ID: {session.session_id}\n")
    
    # Simulate conversation
    queries = [
        "Hi, I prefer communication in a formal tone.",
        "Check my balance for ACC-1001 please.",
        "Can you remember my preference for future conversations?",
    ]
    
    for i, query in enumerate(queries, 1):
        print(f"Customer [{i}]: {query}")
        response = await agent.run(query, session=session)
        print(f"Agent: {response.text}\n")
    
    # Check stored messages
    messages = await redis_provider.get_messages(session.session_id)
    print(f"📊 Messages stored for {customer_id}: {len(messages)}")
    await redis_provider.clear(session.session_id)
    await redis_provider.aclose()
    
    print("✅ Session data cleaned up\n")

await example_user_session_management()

In [ ]:
## Example 3: Conversation Persistence Across Restarts

Simulate resuming a conversation after application restart.

In [6]:
async def example_conversation_persistence():
    """Conversation persistence across app restarts."""
    print("\n--- Conversation Persistence Across Restarts ---\n")
    
    conversation_id = "persistent_banking_session"
    
    # Create agent using FoundryChatClient
    PROJECT_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
    
    client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=AzureCliCredential(),
    )
    
    # Phase 1: Start conversation
    print("=== Phase 1: Starting conversation ===")
    redis_provider1 = RedisHistoryProvider(
        redis_url="redis://localhost:6379",
        ssl=False,
    )
    agent1 = Agent(
        client=client,
        name="BankingSupport",
        instructions="You are a banking support agent.",
        tools=[get_account_balance, transfer_funds],
        context_providers=[redis_provider1],
    )
    
    # Use a fixed session ID so we can resume it
    session1 = AgentSession(session_id=conversation_id)
    
    query1 = "I need to transfer $500 from ACC-1001 to ACC-1002"
    print(f"Customer: {query1}")
    response1 = await agent1.run(query1, session=session1)
    print(f"Agent: {response1.text}")
    messages = await redis_provider1.get_messages(session1.session_id)
    print(f"\n📊 Stored {len(messages)} messages in Redis")
    await redis_provider1.aclose()
    
    # Phase 2: Resume after "restart"
    print("\n=== Phase 2: Resuming after 'app restart' ===")
    redis_provider2 = RedisHistoryProvider(
        redis_url="redis://localhost:6379",
        ssl=False,
    )
    agent2 = Agent(
        client=client,
        name="BankingSupport",
        instructions="You are a banking support agent.",
        tools=[get_account_balance, transfer_funds],
        context_providers=[redis_provider2],
    )
    
    # Resume with the same session ID
    session2 = AgentSession(session_id=conversation_id)
    
    query2 = "What transfer did I just request?"
    print(f"Customer: {query2}")
    response2 = await agent2.run(query2, session=session2)
    print(f"Agent: {response2.text}")
    
    messages = await redis_provider2.get_messages(session2.session_id)
    print(f"\n📊 Total messages after resume: {len(messages)}")
    
    # Cleanup
    await redis_provider2.clear(conversation_id)
    await redis_provider2.aclose()
    
    print("✅ Persistence demo completed\n")

await example_conversation_persistence()


--- Conversation Persistence Across Restarts ---

=== Phase 1: Starting conversation ===
Customer: I need to transfer $500 from ACC-1001 to ACC-1002


ConnectionError: Error Multiple exceptions: [Errno 10061] Connect call failed ('::1', 6379, 0, 0), [Errno 10061] Connect call failed ('127.0.0.1', 6379) connecting to localhost:6379.

## Key Takeaways

| Feature | Description |
|---------|-------------|
| `RedisHistoryProvider` | Built-in Redis storage for conversation history |
| `key_prefix` | Namespace for isolating different customers/sessions |
| `max_messages` | Automatic trimming for memory management |
| `context_providers` | Agent parameter to attach history providers |

## Redis Benefits for FSI

| Benefit | Application |
|---------|-------------|
| **High Performance** | Low-latency customer interactions |
| **Distributed** | Share sessions across app instances |
| **Persistence** | Survive application restarts |
| **Scalability** | Handle high customer volume |